In [1]:
from pathlib import Path
import pandas as pd
BASE = Path.home() / "BigData_UPAO/Osiptel_Proyecto"
excel = BASE / "data/raw/excel/2.1 CONEXIONES DE INTERNET FIJO.xlsx"
archivo = pd.ExcelFile(excel)
print("HOJAS DEL EXCEL:")
for hoja in archivo.sheet_names:
    print("-", hoja)
print("\nVISTA PRELIMINAR:")
muestra = pd.read_excel(
    excel,
    sheet_name=archivo.sheet_names[0],
    header=None,
    nrows=10
)
display(muestra)

HOJAS DEL EXCEL:
- Dataset
- Diccionario

VISTA PRELIMINAR:


,0,1,2,3,4,5,6,7
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,DATASET: CONEXIONES DE INTERNET FIJO,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Mes,Empresa,Tecnologia,Segmento,Departamento,Rango Velocidad de Bajada (nominal),Conexiones
4,NaN,1999-12-01 00:00:00,NaN,1) Dial-Up Fijo,No identificada,NaN,NaN,120014
5,NaN,2000-12-01 00:00:00,NaN,1) Dial-Up Fijo,No identificada,NaN,NaN,130560
6,NaN,2000-12-01 00:00:00,NaN,2) xDSL,No identificada,NaN,NaN,32
7,NaN,2000-12-01 00:00:00,NaN,3) Cablemodem,No identificada,NaN,NaN,788
8,NaN,2001-12-01 00:00:00,NaN,1) Dial-Up Fijo,Comercial,NaN,NaN,3264
9,NaN,2001-12-01 00:00:00,NaN,1) Dial-Up Fijo,Residencial,NaN,NaN,139371


In [3]:
df_excel = pd.read_excel(
    excel,
    sheet_name="Dataset",
    header=3,
    usecols="B:H"
)
print("DIMENSIONES DEL DATASET:")
print(f"Filas: {df_excel.shape[0]}")
print(f"Columnas: {df_excel.shape[1]}")
print("\nCOLUMNAS:")
print(df_excel.columns.tolist())
print("\nTIPOS DE DATOS:")
print(df_excel.dtypes)
print("\nPRIMEROS 5 REGISTROS:")
display(df_excel.head())

DIMENSIONES DEL DATASET:
Filas: 141106
Columnas: 7

COLUMNAS:
['Mes', 'Empresa', 'Tecnologia', 'Segmento', 'Departamento', 'Rango Velocidad de Bajada (nominal)', 'Conexiones']

TIPOS DE DATOS:
Mes                                    datetime64[ns]
Empresa                                        object
Tecnologia                                     object
Segmento                                       object
Departamento                                   object
Rango Velocidad de Bajada (nominal)            object
Conexiones                                      int64
dtype: object

PRIMEROS 5 REGISTROS:


,Mes,Empresa,Tecnologia,Segmento,Departamento,Rango Velocidad de Bajada (nominal),Conexiones
0,1999-12-01,NaN,1) Dial-Up Fijo,No identificada,NaN,NaN,120014
1,2000-12-01,NaN,1) Dial-Up Fijo,No identificada,NaN,NaN,130560
2,2000-12-01,NaN,2) xDSL,No identificada,NaN,NaN,32
3,2000-12-01,NaN,3) Cablemodem,No identificada,NaN,NaN,788
4,2001-12-01,NaN,1) Dial-Up Fijo,Comercial,NaN,NaN,3264


In [4]:
diccionario = pd.read_excel(
    excel,
    sheet_name="Diccionario",
    header=None
)
print("DIMENSIONES DEL DICCIONARIO:")
print(diccionario.shape)
print("\nCONTENIDO DEL DICCIONARIO:")
display(diccionario.head(30))

DIMENSIONES DEL DICCIONARIO:
(17, 4)

CONTENIDO DEL DICCIONARIO:


,0,1,2,3
0,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN
2,NaN,ATRIBUTO,DESCRIPCIÓN,TIPO DE DATO
3,NaN,Mes,Mes al que corresponde el dato reportado.,Fecha: AAAA-MM-DD
4,NaN,Empresa,Nombre (razón social) de la empresa operadora ...,Cadena
5,NaN,Tecnología,Tecnología de acceso por medio del cual se pre...,Cadena
6,NaN,Segmento,Se refiere al segmento al que pertenece la con...,Cadena
7,NaN,Departamento,Departamento donde se encuentra instalada la c...,Cadena
8,NaN,Rango Velocidad de Bajada (nominal),Rango en el que se encuentra la velocidad de b...,Alfanumérico
9,NaN,Conexiones,"Número de conexiones en servicio, al cierre de...",Numérico


In [5]:
print("VALORES NULOS POR COLUMNA:")

nulos = pd.DataFrame({
    "Nulos": df_excel.isnull().sum(),
    "Porcentaje (%)": (
        df_excel.isnull().mean() * 100
    ).round(2)
})
display(nulos)
print("\nFILAS IDENTICAS DUPLICADAS:")
print(df_excel.duplicated().sum())
columnas_clave = [
    col for col in df_excel.columns
    if col != "Conexiones"
]
print("\nCLAVES DE NEGOCIO REPETIDAS:")
print(df_excel.duplicated(subset=columnas_clave).sum())
print("\nVALORES UNICOS POR COLUMNA:")
display(df_excel.nunique(dropna=True))

VALORES NULOS POR COLUMNA:


,Nulos,Porcentaje (%)
Mes,0,0.00
Empresa,91,0.06
Tecnologia,0,0.00
Segmento,0,0.00
Departamento,201,0.14
Rango Velocidad de Bajada (nominal),13029,9.23
Conexiones,0,0.00



FILAS IDENTICAS DUPLICADAS:
2486

CLAVES DE NEGOCIO REPETIDAS:
11722

VALORES UNICOS POR COLUMNA:


Mes                                      71
Empresa                                 161
Tecnologia                                7
Segmento                                  3
Departamento                             28
Rango Velocidad de Bajada (nominal)      14
Conexiones                             8360
dtype: int64

In [6]:
resumen_claves = (
    df_excel.groupby(columnas_clave, dropna=False)
    ["Conexiones"]
    .agg(
        registros="size",
        valores_distintos="nunique"
    )
    .reset_index()
)
repetidas = resumen_claves[
    resumen_claves["registros"] > 1
]
conflictos = repetidas[
    repetidas["valores_distintos"] > 1
]
print("Claves repetidas:", len(repetidas))
print("Claves con conexiones diferentes:", len(conflictos))
display(conflictos.head(5))

Claves repetidas: 1337
Claves con conexiones diferentes: 1289


,Mes,Empresa,Tecnologia,Segmento,Departamento,Rango Velocidad de Bajada (nominal),registros,valores_distintos
12,2002-12-01,NaN,2) xDSL,Comercial,NaN,NaN,4,4
13,2002-12-01,NaN,2) xDSL,Residencial,NaN,NaN,2,2
18,2003-12-01,NaN,2) xDSL,Comercial,NaN,NaN,4,4
19,2003-12-01,NaN,2) xDSL,Residencial,NaN,NaN,2,2
24,2004-12-01,NaN,2) xDSL,Comercial,NaN,NaN,3,3


In [7]:
empresas_excel = (
    df_excel["Empresa"]
    .dropna()
    .astype(str)
    .unique()
)
candidatos = ["MEGACABLE", "TELEF", "ELECTRO DUNAS"]

for nombre in candidatos:
    coincidencias = sorted([
        empresa for empresa in empresas_excel
        if nombre in empresa.upper()
    ])
    print(f"\n{nombre}:")
    print(coincidencias if coincidencias else "Sin coincidencias")
print("\nPERIODO DEL EXCEL:")
print("Desde:", df_excel["Mes"].min())
print("Hasta:", df_excel["Mes"].max())


MEGACABLE:
['MEGACABLE NETWORK S.A.C.']

TELEF:
['TELEFONICA DEL PERU S.A.A.']

ELECTRO DUNAS:
Sin coincidencias

PERIODO DEL EXCEL:
Desde: 1999-12-01 00:00:00
Hasta: 2026-03-01 00:00:00
